# Stacked Bar Chart of Tool Identification in Network Traffic

This notebook loads network traffic CSV files from a specified directory, processes tool identification columns, and visualizes the results as a stacked bar chart. Each bar represents a unique combination of identified tools or 'Not identified' if no tool was detected.

In [ ]:
# Import Required Libraries
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Define the input directory containing the CSV files
input_directory = 'test-tcp-enr-tools'
input_directory = 'test-tcp-enr-tools-linear'
input_directory = 'test-tcp-enr-tools-sliding2'

In [ ]:
tool_columns = ['mirai', 'zmap', 'masscan', 'hajime_possible', 'unicorn']

In [ ]:
def parse_ibr_info(val):
    try:
        return json.loads(val) if isinstance(val, str) else (val if isinstance(val, dict) else {})
    except Exception:
        return {}

In [ ]:
# Function to load CSV files from a directory
def load_csv(file):
    columns = [
        "timestamp","ip_version","ip_orig_tos","ip_tos","ip_prec","ip_dscp","ip_enc","ip_len","ip_id","ip_ttl","ip_chksum",
        "ip_src","ip_dst","ip_options","tcp_sport","tcp_dport","tcp_seq","tcp_ack","tcp_dataofs","tcp_reserved","tcp_orig_flags",
        "tcp_CWR","tcp_ACK","tcp_PSH","tcp_RST","tcp_SYN","tcp_URG","tcp_FIN","tcp_ECE","tcp_NS","tcp_other_flags","tcp_window",
        "tcp_chksum","tcp_urgptr","tcp_options","payload_hex","key","IBR_info"
    ]

    df = pd.read_csv(file, usecols=columns)

    # Parse IBR_info as dict for each row
    df['IBR_info'] = df['IBR_info'].apply(parse_ibr_info)

    return df


In [ ]:
files = glob.glob(os.path.join(input_directory, "*.csv"))
files = sorted(files)

In [ ]:
def convert_df_columns_to_bool(df):
    # Now tool columns are inside IBR_info, so extract them
    for col in tool_columns:
        df[col] = df['IBR_info'].apply(lambda x: x.get(col, False))
    return df

In [ ]:
# This function aggregates the tool columns
def get_columns_agg_counts(df):
    df = convert_df_columns_to_bool(df)
    df = df[tool_columns]
    df = df.copy()
    df_agg = df.groupby(tool_columns).size().reset_index(name='count')
    return df_agg


In [ ]:
# Final dataframe to hold all aggregated data
final_df = pd.DataFrame()

for file in files:
    df = load_csv(file)
    df_agg = get_columns_agg_counts(df)

    # Get the date from the filename
    date_str = os.path.basename(file).split('_')[1]

    # Add the date column to the aggregated dataframe
    df_agg['date'] = date_str

    print(f"Processed file: {file} with {len(df_agg)} aggregated rows.")
    final_df = pd.concat([final_df, df_agg], ignore_index=True)

In [ ]:
# Create the tool_label column in final_df, renaming 'hajime_possible' to 'hajime' in the labels
def get_tool_label(row):
    def label_map(col):
        if col == 'hajime_possible':
            return 'Hajime'
        return col.capitalize()
    active = [label_map(col) for col in tool_columns if row[col]]
    if not active:
        return 'Not identified'
    return '+'.join(sorted(active))

final_df['tool_label'] = final_df.apply(get_tool_label, axis=1)

In [ ]:
# Plot the stacked bar chart of tool identification by day
pivot_df = final_df.pivot_table(index='date', columns='tool_label', values='count', aggfunc='sum', fill_value=0)
pivot_df.plot(kind='bar', stacked=True, figsize=(12,8), colormap='tab20')
plt.xlabel('Date')
plt.ylabel('Packets')
plt.title('Stacked Bar Chart of Tool Identification by Day')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Tool(s) identified', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
def remove_ambiguous_and_sum_nonidentified(df, tool_columns, date_col='date', count_col='count'):
    df = df.copy()
    # Identify ambiguous rows (more than one tool is True)
    ambiguous_mask = df[tool_columns].sum(axis=1) > 1
    # Group the count of ambiguous rows by day
    non_identified = (
        df.loc[ambiguous_mask]
        .groupby(date_col)[count_col]
        .sum()
        .reset_index()
        .assign(tool_label='Not identified')
    )
    # Filter non-ambiguous rows
    clean_df = df.loc[~ambiguous_mask].copy()
    # Add the "Not identified" rows
    result = pd.concat([clean_df, non_identified], ignore_index=True)
    return result

In [ ]:
def remove_ambiguous_and_sum_nonidentified_unicorn(df, tool_columns, date_col='date', count_col='count'):
    df = df.copy()
    # Create mirai_unicorn column to identify rows where both are True
    df['mirai_unicorn'] = (df['mirai'] & df['unicorn'])
    # Identify ambiguous rows (more than one tool is True, except mirai+unicorn)
    ambiguous_mask = (df[tool_columns].sum(axis=1) > 1) & (~df['mirai_unicorn'])
    # Group the count of ambiguous rows by day
    non_identified = (
        df.loc[ambiguous_mask]
        .groupby(date_col)[count_col]
        .sum()
        .reset_index()
        .assign(tool_label='Mirai+Unicorn')
    )
    # Filter non-ambiguous rows and mirai+unicorn rows
    clean_df = df.loc[~ambiguous_mask].copy()
    # Assign special label to mirai+unicorn
    clean_df.loc[clean_df['mirai_unicorn'], 'tool_label'] = 'Mirai+Unicorn'
    # Add the "Mirai+Unicorn" rows (previously was Not identified)
    result = pd.concat([clean_df, non_identified], ignore_index=True)
    return result


In [ ]:
df_agg_cleaned = remove_ambiguous_and_sum_nonidentified(final_df, tool_columns)

In [ ]:
df_agg_cleaned = remove_ambiguous_and_sum_nonidentified_unicorn(final_df, tool_columns)

In [ ]:
def plot_stacked_bar_chart(df, colormap='tab10', font_size=14):
    import pandas as pd
    # Plot stacked bars by day and tool type
    pivot_df = df.pivot_table(index='date', columns='tool_label', values='count', aggfunc='sum', fill_value=0)
    # Order columns so 'Masscan' is at the top
    cols = list(pivot_df.columns)
    if 'Masscan' in cols:
        cols = [c for c in cols if c != 'Masscan'] + ['Masscan']
        pivot_df = pivot_df[cols]
    # And "Not identified" at the bottom
    if 'Not identified' in cols:
        cols = ['Not identified'] + [c for c in cols if c != 'Not identified']
        pivot_df = pivot_df[cols]
    # Convert date index to YYYY-MM-DD format
    try:
        pivot_df.index = pd.to_datetime(pivot_df.index, errors='coerce').strftime('%Y-%m-%d')
    except Exception:
        pass
    ax = pivot_df.plot(kind='bar', stacked=True, figsize=(12,8), colormap=colormap)
    plt.xlabel('Date', fontsize=font_size)
    plt.ylabel('Packets', fontsize=font_size)
    plt.xticks(rotation=45, ha='right', fontsize=font_size)
    plt.yticks(fontsize=font_size)
    plt.title('Stacked Bar Chart of Tool Identification by Day', fontsize=font_size)
    ax.legend(title='Tool(s) identified', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=font_size, title_fontsize=font_size)
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_stacked_bar_chart_b(df, colormap='tab10', font_size=18, legend_borderpad=0.2, legend_labelspacing=0.2, legend_handletextpad=0.2):
    # Plot stacked bars by day and tool type
    pivot_df = df.pivot_table(index='date', columns='tool_label', values='count', aggfunc='sum', fill_value=0)
    # Order columns so 'Masscan' is at the top
    cols = list(pivot_df.columns)
    if 'Masscan' in cols:
        cols = [c for c in cols if c != 'Masscan'] + ['Masscan']
        pivot_df = pivot_df[cols]

    # And "Not identified" at the bottom
    if 'Not identified' in cols:
        cols = ['Not identified'] + [c for c in cols if c != 'Not identified']
        pivot_df = pivot_df[cols]

    # Convert date index to YYYY-MM-DD format
    try:
        pivot_df.index = pd.to_datetime(pivot_df.index, errors='coerce').strftime('%Y-%m-%d')
    except Exception:
        pass

    ax = pivot_df.plot(kind='bar', stacked=True, figsize=(12,8), colormap=colormap)
    plt.xlabel('Date', fontsize=font_size)
    plt.ylabel('Packets (millions)', fontsize=font_size)
    # Show only every other date on the X axis
    xticks = ax.get_xticks()
    xlabels = [label.get_text() for label in ax.get_xticklabels()]
    new_labels = [lbl if i % 2 == 0 else '' for i, lbl in enumerate(xlabels)]
    ax.set_xticklabels(new_labels, rotation=45, ha='right', fontsize=font_size)
    # Format yticks in millions with one decimal
    yticks = ax.get_yticks()
    ylabels = [f"{y/1e6:.1f}" for y in yticks]
    ax.set_yticklabels(ylabels, fontsize=font_size)
    # Legend above the plot, in three columns, no title, reduced padding
    leg = ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.14), ncol=4, fontsize=font_size, title=None,
                    borderpad=legend_borderpad, labelspacing=legend_labelspacing, handletextpad=legend_handletextpad)
    plt.tight_layout(pad=0.5)
    plt.savefig('tcptools.pdf')
    plt.show()

In [ ]:
plot_stacked_bar_chart(final_df, colormap='tab10')

In [ ]:
plot_stacked_bar_chart_b(df_agg_cleaned, colormap='tab10')

In [ ]:
final_df[final_df['date'] == '20251011']['count'].sum()

In [ ]:
df_agg_cleaned[df_agg_cleaned['date'] == '20251011']['count'].sum()